# Materials 05 — Fracture of a grain boundary

Grain boundaries are frequently where materials fail. Following the original
Tutorial 5, we pull a bicrystal apart **perpendicular to its boundary** until
it fractures, quasi-statically: rigid *grips* at top and bottom are displaced
by small increments, and the interior is re-minimized after each step — an
athermal, ideal-strength test (contrast with the finite-temperature dynamics
of [03](03-uniaxial-deformation.ipynb)).

We reuse the Σ5(310) bicrystal from [04](04-grain-boundary.ipynb), now with
**free surfaces** along y (`boundary p s p`) so there is a single boundary in
the middle of a free-standing slab.

In [ ]:
%pip install lammps-js matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lammps import lammps

A0 = 1.5496
P = A0 * np.sqrt(10) / 2
LX, LY, LZ = 4 * P, 5 * P, 2 * A0

lmp = await lammps(output=None)
lmp.commands_string(f"""
units lj
atom_style atomic
boundary p s p
lattice fcc {4 / A0**3:.8f}
region box block 0 {LX:.6f} -{LY:.6f} {LY:.6f} 0 {LZ:.6f} units box
create_box 2 box
lattice fcc {4 / A0**3:.8f} orient x 0 3 1 orient y 0 -1 3 orient z 1 0 0
region upper block INF INF 0.0 INF INF INF units box
create_atoms 1 region upper
lattice fcc {4 / A0**3:.8f} orient x 0 3 -1 orient y 0 1 3 orient z 1 0 0
region lower block INF INF INF 0.0 INF INF units box
create_atoms 2 region lower
mass * 1.0
pair_style lj/cut 2.5
pair_coeff * * 1.0 1.0 2.5
delete_atoms overlap 0.7 all all
min_style cg
minimize 1.0e-9 1.0e-9 2000 20000

variable ytop equal bound(all,ymax)
variable ybot equal bound(all,ymin)
region rtop block INF INF $(v_ytop-1.6) INF INF INF units box
region rbot block INF INF INF $(v_ybot+1.6) INF INF units box
group gtop region rtop
group gbot region rbot
fix holdtop gtop setforce 0.0 0.0 0.0
fix holdbot gbot setforce 0.0 0.0 0.0
variable ftop equal f_holdtop[2]
""")
print(lmp.get_natoms(), "atoms in the bicrystal slab")

`fix setforce 0` freezes the grip atoms *and reports the total force it
zeroed out* — that reaction force, divided by the cross-section, is exactly
the tensile stress carried by the sample.

## Load until it breaks

Each iteration: displace the grips apart by 0.04 σ each, re-minimize the
interior, record the reaction force:

In [ ]:
area = lmp.get_thermo("lx") * lmp.get_thermo("lz")
strain_step = 2 * 0.04 / (2 * LY)
curve = []
for i in range(28):
    lmp.commands_string("""
displace_atoms gtop move 0.0 0.04 0.0 units box
displace_atoms gbot move 0.0 -0.04 0.0 units box
minimize 1.0e-9 1.0e-9 500 5000
""")
    curve.append(-lmp.extract_variable("ftop") / area)
curve = np.array(curve)
strain = strain_step * np.arange(1, len(curve) + 1)
peak = curve.argmax()
print(f"peak stress {curve[peak]:.2f} eps/sigma^3 at strain {strain[peak]:.3f}")

In [ ]:
plt.figure(figsize=(5.5, 3.6))
plt.plot(strain, curve, "o-", color="tab:red")
plt.axhline(0, lw=0.5, color="gray")
plt.xlabel("engineering strain"); plt.ylabel("stress σᵧᵧ (ε/σ³)")
plt.title("Quasi-static tension across the Σ5(310) boundary")
plt.tight_layout(); plt.show()

The stress rises, then **drops abruptly** — the boundary cleaves. This peak
is the *ideal* (athermal, defect-free) cleavage strength: with no
pre-existing cracks or thermal fluctuations, the boundary can be stretched
all the way to its cohesive limit. Real materials contain flaws that
concentrate stress and fracture far earlier.

## Watch the crack

Project the final configuration on the x–y plane: the sample has separated —
and if it cleaved at the boundary, the two halves are single crystals:

In [ ]:
x = lmp.extract_atom("x")
types = lmp.extract_atom("type")
plt.figure(figsize=(6, 5))
plt.scatter(x[:, 0], x[:, 1], c=types, s=25, cmap="coolwarm")
plt.xlabel("x (σ)"); plt.ylabel("y (σ)")
plt.title("After fracture (color = original grain)")
plt.gca().set_aspect("equal")
plt.tight_layout(); plt.show()

lmp.close()

**Exercises**
- Where did it break — at the boundary (colors separate cleanly) or inside a
  grain? Rerun with `delete_atoms overlap 0.3` (the high-energy boundary from
  [04](04-grain-boundary.ipynb)): a worse boundary should fail earlier.
- Reduce the displacement increment. The peak stress converges; the
  post-peak drop stays abrupt — brittle fracture is intrinsically sudden.

Next: [06 — Nanoindentation](06-nanoindentation.ipynb): probing a surface by
poking it.